In [65]:
from google.cloud import bigquery
import os
import plotly.express as px
import pandas as pd
from pyproj import Transformer

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../data/auth/sante-et-territoires-216daad01024.json" 
client = bigquery.Client() 

query = """
SELECT *
FROM `sante-et-territoires.finess.soin_etab`
"""

df = client.query(query).to_dataframe()
df_communes = pd.read_csv("../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8")



/tmp/ipykernel_43809/3866254769.py:16: DtypeWarning: Columns (1,9,11,13,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_communes = pd.read_csv("../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8")


In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10291 entries, 0 to 10290
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   nofinesset      10291 non-null  object
 1   rsej            10291 non-null  object
 2   activite        10291 non-null  object
 3   libactivite     10291 non-null  object
 4   forme           10291 non-null  object
 5   libforme        10291 non-null  object
 6   nofinessej      10291 non-null  object
 7   rs              10291 non-null  object
 8   rslongue        8832 non-null   object
 9   commune         10291 non-null  object
 10  departement     10291 non-null  object
 11  libdepartement  10291 non-null  object
 12  categetab       10291 non-null  object
 13  libcategetab    10291 non-null  object
 14  coordyet        10291 non-null  object
 15  coordxet        10291 non-null  object
dtypes: object(16)
memory usage: 1.3+ MB


In [67]:

# Création du transformateur Lambert 93 -> WGS84
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

# Conversion
df["longitude"], df["latitude"] = transformer.transform(
    df["coordxet"].values,df["coordyet"].values
)

# Aperçu
print(df[["nofinesset", "coordxet", "coordyet", "latitude", "longitude"]].head())

  nofinesset   coordxet   coordyet   latitude  longitude
0  060024411  1041862.7  6297002.1  43.689403   7.241235
1  060785003  1044997.0  6301126.4  43.724939   7.282816
2  060785003  1044997.0  6301126.4  43.724939   7.282816
3  060785003  1044997.0  6301126.4  43.724939   7.282816
4  060785003  1044997.0  6301126.4  43.724939   7.282816


In [68]:
df.head()

,nofinesset,rsej,activite,libactivite,forme,libforme,nofinessej,rs,rslongue,commune,departement,libdepartement,categetab,libcategetab,coordyet,coordxet,longitude,latitude
0,060024411,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,14,Médecine d'urgence,14,Non saisonnier,060785011,CHU DE NICE ANTENNE SMUR SITE HPNCL,CHU NICE ANTENNE SMUR SITE HOP PEDIATRIQUES NI...,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6297002.1,1041862.7,7.241235,43.689403
1,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,01,Médecine,01,Hospitalisation complète (24 heures consécutiv...,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939
2,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,16,Traitement de l'insuffisance rénale chronique ...,00,Pas de forme,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939
3,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,80,Greffe de rein,00,Pas de forme,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939
4,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,16,Traitement de l'insuffisance rénale chronique ...,00,Pas de forme,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939


In [69]:
df['libactivite'].value_counts()

libactivite
Médecine                                                                                                                                 1531
Traitement de l'insuffisance rénale chronique par épuration extrarénale                                                                  1290
Psychiatrie                                                                                                                              1212
Médecine d'urgence                                                                                                                       1105
AMP DPN                                                                                                                                   911
Traitement du cancer                                                                                                                      878
Soins de suite et de réadaptation non spécialisés                                                                                       

In [70]:
# Regroupement des catégories d'activités
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["greffe"]):
        return "Greffe"

    if any(x in cat for x in ["soins de suite"]):
        return "Soins de suite"

    if any(x in cat for x in ["chirurgie"]):
        return "Chirurgie"
    if any(x in cat for x in ["médecine d\'urgence"]):
        return "Médecine d'urgence"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["psychiatrie"]):
        return "Psychiatrie"
    if any(x in cat for x in ["amp dpn"]):
        return "AMP DPN"
    if any(x in cat for x in ["cancer"]):
        return "Cancer"
    if any(x in cat for x in ["gynécologie"]):
        return "Gynécologie"
    if any(x in cat for x in ["soins de longue durée"]):
        return "Soins de longue durée"
    if any(x in cat for x in ["insuffisance rénale"]):
        return "Insuffisance rénale"
    if any(x in cat for x in ["examen des caractéristiques génétiques"]):
        return "Examen des caractéristiques génétiques"
    return "Autres"

# Application
df["groupe_activites"] = df["libactivite"].apply(regrouper_categorie)

In [71]:
df['groupe_activites'].value_counts()

groupe_activites
Médecine                                  1531
Soins de suite                            1319
Insuffisance rénale                       1290
Psychiatrie                               1212
Médecine d'urgence                        1105
AMP DPN                                    911
Cancer                                     878
Gynécologie                                757
Chirurgie                                  383
Soins de longue durée                      381
Autres                                     204
Greffe                                     180
Examen des caractéristiques génétiques     140
Name: count, dtype: int64

In [72]:
df.loc[df['groupe_activites'] == "Autres", 'libactivite']


6        Activités interventionnelles sous imagerie méd...
65                            Traitement des grands brûlés
68                            Traitement des grands brûlés
71       Activités interventionnelles sous imagerie méd...
109      Activités interventionnelles sous imagerie méd...
                               ...                        
10244    Activités interventionnelles sous imagerie méd...
10245    Activités interventionnelles sous imagerie méd...
10246    Activités interventionnelles sous imagerie méd...
10265    Activités interventionnelles sous imagerie méd...
10269    Activités interventionnelles sous imagerie méd...
Name: libactivite, Length: 204, dtype: object

In [73]:
#nettoyage des doublons
df_clean = df.drop_duplicates()

In [74]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7770 entries, 0 to 10290
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   nofinesset        7770 non-null   object 
 1   rsej              7770 non-null   object 
 2   activite          7770 non-null   object 
 3   libactivite       7770 non-null   object 
 4   forme             7770 non-null   object 
 5   libforme          7770 non-null   object 
 6   nofinessej        7770 non-null   object 
 7   rs                7770 non-null   object 
 8   rslongue          6665 non-null   object 
 9   commune           7770 non-null   object 
 10  departement       7770 non-null   object 
 11  libdepartement    7770 non-null   object 
 12  categetab         7770 non-null   object 
 13  libcategetab      7770 non-null   object 
 14  coordyet          7770 non-null   object 
 15  coordxet          7770 non-null   object 
 16  longitude         7770 non-null   float64
 17 

In [75]:
#je crée un colonne code_insee à partir de la colonne code commune et de la colonne departement
df_clean.loc[:, "code_insee"] = (
    df_clean["departement"].astype(str).str.zfill(2)
    + df_clean["commune"].astype(str).str.zfill(3)
)

df_clean.head()

/tmp/ipykernel_43809/2999015001.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.loc[:, "code_insee"] = (


,nofinesset,rsej,activite,libactivite,forme,libforme,nofinessej,rs,rslongue,commune,departement,libdepartement,categetab,libcategetab,coordyet,coordxet,longitude,latitude,groupe_activites,code_insee
0,060024411,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,14,Médecine d'urgence,14,Non saisonnier,060785011,CHU DE NICE ANTENNE SMUR SITE HPNCL,CHU NICE ANTENNE SMUR SITE HOP PEDIATRIQUES NI...,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6297002.1,1041862.7,7.241235,43.689403,Médecine d'urgence,06088
1,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,01,Médecine,01,Hospitalisation complète (24 heures consécutiv...,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939,Médecine,06088
2,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,16,Traitement de l'insuffisance rénale chronique ...,00,Pas de forme,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939,Insuffisance rénale,06088
3,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,80,Greffe de rein,00,Pas de forme,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939,Greffe,06088
5,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,12,Neurochirurgie,00,Pas de forme,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939,Chirurgie,06088


In [76]:
# latitudes longitudes en float
df_clean.loc[:,'latitude'] = df_clean['latitude'].astype(float)
df_clean.loc[:,'longitude'] = df_clean['longitude'].astype(float)

In [77]:
df_clean = df_clean.rename(columns={
    "rs": "raison_sociale",
    "libcategetab": "categorie",
    "commune": "code_commune",
    "nofinessej": "numero finess etablissement juridique",
      'nofinesset': "numero_finess_etablissement", 
        'libdepartement': "libelle_departement",
         'groupe' : "type d etablissements",
})

In [78]:
#je filtre les données sur les départements de la région Occitanie
departements_occitanie = ["09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82",  ]
df_occitanie = df_clean[df_clean["departement"].isin(departements_occitanie)]

In [79]:
df_occitanie.head()

,numero_finess_etablissement,rsej,activite,libactivite,forme,libforme,numero finess etablissement juridique,raison_sociale,rslongue,code_commune,departement,libelle_departement,categetab,categorie,coordyet,coordxet,longitude,latitude,groupe_activites,code_insee
245,300016680,CHU NIMES,04,Psychiatrie,03,Hospitalisation à temps partiel de jour,300780038,HJ PIJ CMPEA OUEST MONTAURY CHU NIMES,HJ PSY INFANTO JUVENILE ET CMPEA OUEST MONTAUR...,189,30,GARD,101,Centre Hospitalier Régional (C.H.R.),6304861.0,808372.0,4.347319,43.834439,Psychiatrie,30189
246,300016698,CHU NIMES,04,Psychiatrie,03,Hospitalisation à temps partiel de jour,300780038,HJ PIJ CMPEA VAUVERT CHU NIMES,HJ PSY INFANTO JUVENILE CMPEA VAUVERT,341,30,GARD,101,Centre Hospitalier Régional (C.H.R.),6289570.7,803332.6,4.281594,43.697621,Psychiatrie,30341
247,300017910,CHU NIMES,04,Psychiatrie,03,Hospitalisation à temps partiel de jour,300780038,HJ PSY GEN BOURDALOUE CHU NIMES,HJ PSY ADULTES BOURDALOUE CHU NIMES,189,30,GARD,101,Centre Hospitalier Régional (C.H.R.),6304644.3,809551.7,4.361936,43.832307,Psychiatrie,30189
248,300782117,CHU NIMES,14,Médecine d'urgence,00,Pas de forme,300780038,CHU NIMES CAREMEAU,GROUPE HOPITALIER CAREMEAU CHU NIMES TERRITOIR...,189,30,GARD,101,Centre Hospitalier Régional (C.H.R.),6303501.8,806105.0,4.318858,43.822554,Médecine d'urgence,30189
249,300782117,CHU NIMES,18,Traitement du cancer,00,Pas de forme,300780038,CHU NIMES CAREMEAU,GROUPE HOPITALIER CAREMEAU CHU NIMES TERRITOIR...,189,30,GARD,101,Centre Hospitalier Régional (C.H.R.),6303501.8,806105.0,4.318858,43.822554,Cancer,30189


In [80]:
#j'exporte en csv dans processed l occitanie plus les départements limitrophes
departement_limitrophes_et_occitanie= ["64", "40", "47", "24", "19", "15", "43", "07", "84", "13", "09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82"]
df_limit= df_clean[df_clean["departement"].isin(departement_limitrophes_et_occitanie)]
df_limit.info()
df_limit.to_csv("../data/processed/soins_limit.csv", index=False, sep=";", encoding="utf-8")

<class 'pandas.core.frame.DataFrame'>
Index: 1756 entries, 47 to 10280
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   numero_finess_etablissement            1756 non-null   object 
 1   rsej                                   1756 non-null   object 
 2   activite                               1756 non-null   object 
 3   libactivite                            1756 non-null   object 
 4   forme                                  1756 non-null   object 
 5   libforme                               1756 non-null   object 
 6   numero finess etablissement juridique  1756 non-null   object 
 7   raison_sociale                         1756 non-null   object 
 8   rslongue                               1746 non-null   object 
 9   code_commune                           1756 non-null   object 
 10  departement                            1756 non-null   object 
 11  libelle

In [81]:
#j'exporte en csv dans processed la partie occitanie

df_occitanie.to_csv("../data/processed/soins_occitanie.csv", index=False, sep=";", encoding="utf-8")


In [82]:
# Je crée un dataframe pour les urgences uniquement 
df_urgences_france = df_clean[df_clean['groupe_activites'].str.contains("urgence", case=False, na=False)]

In [83]:
df_urgences.head()

,nofinesset,rsej,activite,libactivite,forme,libforme,nofinessej,rs,rslongue,commune,departement,libdepartement,categetab,libcategetab,coordyet,coordxet,longitude,latitude,groupe_activites,code_insee
0,060024411,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,14,Médecine d'urgence,14,Non saisonnier,060785011,CHU DE NICE ANTENNE SMUR SITE HPNCL,CHU NICE ANTENNE SMUR SITE HOP PEDIATRIQUES NI...,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6297002.1,1041862.7,7.241235,43.689403,Médecine d'urgence,06088
7,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,14,Médecine d'urgence,00,Pas de forme,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939,Médecine d'urgence,06088
12,060785003,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,14,Médecine d'urgence,14,Non saisonnier,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6301126.4,1044997.0,7.282816,43.724939,Médecine d'urgence,06088
43,060789195,CTRE HOSPITALIER UNIVERSITAIRE DE NICE,14,Médecine d'urgence,14,Non saisonnier,060785011,CHU DE NICE HOPITAL DE L'ARCHET,CHU DE NICE HOPITAL DE L'ARCHET,088,06,ALPES MARITIMES,101,Centre Hospitalier Régional (C.H.R.),6298190.5,1038763.1,7.203637,43.701572,Médecine d'urgence,06088
47,130044977,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR DE MARIGNANE,APHM ANTENNE SMUR DE MARIGNANE SITE SDIS,054,13,BOUCHES DU RHONE,101,Centre Hospitalier Régional (C.H.R.),6260484.0,879240.3,5.212552,43.420867,Médecine d'urgence,13054


Fonction Haversine (distance en km)
🌍 Principe général
La Terre est (approximativement) une sphère.
Pour calculer la distance entre deux points sur une sphère, on ne peut pas utiliser la distance euclidienne classique.
La formule de Haversine permet de calculer la distance orthodromique, c’est‑à‑dire la distance la plus courte entre deux points à la surface de la Terre.

In [84]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [85]:
#filtre sur les communes de l'occitanie
df_communes_occitanie = df_communes[df_communes['reg_nom'] == 'Occitanie']

In [86]:
# On crée une matrice de distances
distances = []

for idx, commune in df_communes_occitanie.iterrows():
    lat_c, lon_c = commune['latitude_centre'], commune['longitude_centre']
    
    # distances entre la commune et tous les établissements d'urgence
    d = haversine(
        lat_c, lon_c,
        df_urgences['latitude'].values,
        df_urgences['longitude'].values
    )
    
    # distance minimale = établissement le plus proche
    distances.append(d.min())

df_communes_occitanie['distance_urgence_km'] = distances


/tmp/ipykernel_43809/795381311.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_communes_occitanie['distance_urgence_km'] = distances


In [87]:
#csv des distances des communes aux urgences en occitanie
df_communes_occitanie.to_csv("../data/processed/distances_communes_urgence_occitanie.csv", index=False)
